# 🌊 Tree-Ring Watermarks: Baseline vs Optimized

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashhad1130/Invisible_Watermark_GNN/blob/main/Google_Colab_TreeRing_Watermark.ipynb)

**Paper:** [Tree-Ring Watermarks: Fingerprints for Diffusion Images that are Invisible and Robust](https://arxiv.org/abs/2305.20030)

This notebook runs the full Tree-Ring Watermark pipeline on **Google Colab (GPU)** or locally on **Apple Silicon (MPS)**.

## What this notebook does
1. 📦 Installs dependencies
2. 📂 Clones the required repositories
3. 🖥️ Checks your GPU / device
4. 🤗 Downloads Stable Diffusion v1.5 (~2 GB, cached after first run)
5. 🔬 Runs the **baseline** experiment (original paper settings)
6. ⚡ Runs the **optimized** experiment (multi-channel, larger radius, amplitude scaling)
7. 📊 Compares results with a table, bar charts, and sample images
8. 💾 Optionally saves everything to Google Drive

## Before you start
> **Runtime → Change runtime type → GPU** (T4 is free; A100 is faster on Colab Pro)

| Scale | Images | Attacks | Est. time (T4) |
|-------|--------|---------|----------------|
| small | 10     | 7       | ~15–25 min     |
| large | 100    | 14      | ~3–5 hours     |

---
## ⚙️ Cell 1 — Configuration
Adjust these settings before running the rest of the notebook.

In [ ]:
#@title Experiment settings { display-mode: "form" }
SCALE = "small"  #@param ["small", "large"]
SKIP_CLIP = True  #@param {type:"boolean"}
#@markdown *(Skip CLIP scores to save ~3 GB download and ~30 % runtime)*
SCALE_FACTOR = 1.1  #@param {type:"number"}
#@markdown *(Amplitude scale for the optimized watermark injection; 1.0 = disabled)*

import os
print(f"Scale        : {SCALE}")
print(f"Skip CLIP    : {SKIP_CLIP}")
print(f"Scale factor : {SCALE_FACTOR}")

---
## 📦 Cell 2 — Install Dependencies
Colab ships with a compatible PyTorch + CUDA; we only need to add the diffusion-specific packages.

> **Run once per session.** If you restart the runtime, run this cell again.

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:] if result.stdout else "")
        print(result.stderr[-2000:] if result.stderr else "")
        raise RuntimeError(f"Command failed: {cmd}")

# Diffusion / ML packages (torch & torchvision are pre-installed on Colab)
print("Installing diffusers, transformers and helpers...")
run(
    'pip install -q '
    'diffusers==0.21.4 '
    'transformers==4.35.2 '
    'accelerate '
    'safetensors '
    '"huggingface-hub>=0.19,<0.25" '
    'datasets '
    'scipy '
    'scikit-learn '
    'open-clip-torch '
    'Pillow '
    'matplotlib '
    'tqdm '
    '"numpy<2.0"'
)
print("✅ All packages installed.")

---
## 📂 Cell 3 — Clone Repositories
We need two repos:
1. **`Ashhad1130/Invisible_Watermark_GNN`** — this project's experiment code
2. **`YuxinWenRick/tree-ring-watermark`** — the original paper's inference code

In [ ]:
import os, sys
from pathlib import Path

# ── paths ──────────────────────────────────────────────────────────────────
WORK_DIR = Path("/content")
PROJECT_DIR = WORK_DIR / "Invisible_Watermark_GNN"
TREE_RING_DIR = PROJECT_DIR / "tree-ring-watermark"
EXPERIMENT_DIR = PROJECT_DIR / "experiment"

# ── clone this project ─────────────────────────────────────────────────────
if not PROJECT_DIR.exists():
    print("Cloning Invisible_Watermark_GNN...")
    os.system(
        f"git clone --depth 1 "
        f"https://github.com/Ashhad1130/Invisible_Watermark_GNN.git "
        f"{PROJECT_DIR}"
    )
    print("  Done.")
else:
    print(f"  {PROJECT_DIR} already exists — pulling latest changes.")
    os.system(f"git -C {PROJECT_DIR} pull --ff-only")

# ── clone the original tree-ring repo ─────────────────────────────────────
if not TREE_RING_DIR.exists():
    print("Cloning tree-ring-watermark...")
    os.system(
        f"git clone --depth 1 "
        f"https://github.com/YuxinWenRick/tree-ring-watermark.git "
        f"{TREE_RING_DIR}"
    )
    print("  Done.")
else:
    print(f"  {TREE_RING_DIR} already exists — skipping clone.")

# ── create output directories ──────────────────────────────────────────────
for sub in ["checkpoints", "results", "outputs"]:
    (EXPERIMENT_DIR / sub).mkdir(parents=True, exist_ok=True)

# ── add to Python path ─────────────────────────────────────────────────────
for p in [str(TREE_RING_DIR), str(EXPERIMENT_DIR), str(PROJECT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(PROJECT_DIR)
print(f"\nWorking directory : {Path.cwd()}")
print(f"tree-ring repo    : {TREE_RING_DIR}")
print(f"experiment dir    : {EXPERIMENT_DIR}")
print("✅ Repositories ready.")

---
## 🖥️ Cell 4 — Check GPU / Device
Make sure you have a GPU runtime selected (**Runtime → Change runtime type → GPU**).

In [ ]:
import torch
import diffusers, transformers

print(f"PyTorch     : {torch.__version__}")
print(f"diffusers   : {diffusers.__version__}")
print(f"transformers: {transformers.__version__}")

if torch.cuda.is_available():
    DEVICE = "cuda"
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"\n✅ GPU  : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM : {vram_gb:.1f} GB")
    print(f"   CUDA : {torch.version.cuda}")
    if vram_gb < 10:
        print("⚠️  Less than 10 GB VRAM detected. Consider using a higher-tier GPU.")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("\n✅ Device: Apple Silicon MPS")
else:
    DEVICE = "cpu"
    print("\n⚠️  No GPU found — running on CPU (very slow).")
    print("   Go to Runtime → Change runtime type → GPU to enable GPU acceleration.")

print(f"\nDevice selected: {DEVICE}")

---
## 🤗 Cell 5 — Download Stable Diffusion v1.5
Downloads ~2 GB once, then caches to `~/.cache/huggingface`. Subsequent runs are instant.

In [ ]:
import sys
from pathlib import Path

# Ensure paths are set (re-run if kernel restarted after install)
PROJECT_DIR = Path("/content/Invisible_Watermark_GNN")
TREE_RING_DIR = PROJECT_DIR / "tree-ring-watermark"
EXPERIMENT_DIR = PROJECT_DIR / "experiment"
for p in [str(TREE_RING_DIR), str(EXPERIMENT_DIR), str(PROJECT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from diffusers import DPMSolverMultistepScheduler
from inverse_stable_diffusion import InversableStableDiffusionPipeline
import torch

MODEL_ID = "runwayml/stable-diffusion-v1-5"
print(f"Downloading / verifying {MODEL_ID} ...")

scheduler = DPMSolverMultistepScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
_pipe = InversableStableDiffusionPipeline.from_pretrained(
    MODEL_ID, scheduler=scheduler, torch_dtype=torch.float16
)
del _pipe  # free memory — will reload properly in the experiment runners
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"✅ Model cached. Ready to run experiments.")

---
## 🔬 Cell 6 — Run BASELINE Experiment
Uses the original Tree-Ring paper settings:
- Single-channel watermarking (`w_channel=3`)
- Ring radius 10
- 50 DDIM inversion steps

Results and checkpoints are saved to `experiment/results/` and `experiment/checkpoints/`.  
Interrupted runs automatically resume from the last checkpoint.

In [ ]:
import sys, importlib
from pathlib import Path

PROJECT_DIR = Path("/content/Invisible_Watermark_GNN")
TREE_RING_DIR = PROJECT_DIR / "tree-ring-watermark"
EXPERIMENT_DIR = PROJECT_DIR / "experiment"
for p in [str(TREE_RING_DIR), str(EXPERIMENT_DIR), str(PROJECT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import experiment.configs as configs
import experiment.run_baseline as run_baseline_mod
importlib.reload(configs)
importlib.reload(run_baseline_mod)

config_bl = (
    configs.get_small_scale_baseline()
    if SCALE == "small"
    else configs.get_large_scale_baseline()
)

print(f"Running BASELINE ({SCALE} scale, {config_bl.end - config_bl.start} images, "
      f"{len(config_bl.attacks)} attacks) ...")
# results are also saved to experiment/results/ as JSON
baseline_results = run_baseline_mod.run_baseline(config_bl, skip_clip=SKIP_CLIP)
print("\n✅ Baseline experiment complete.")

---
## ⚡ Cell 7 — Run OPTIMIZED Experiment
Enhancements over the baseline:
1. **Multi-channel watermarking** (`w_channel=-1`): all 4 latent channels
2. **Larger radius** (`w_radius=14`): more frequency coefficients carry the mark
3. **More inversion steps** (`test_steps=75`): better DDIM inversion accuracy
4. **Amplitude-scaled injection** (`×SCALE_FACTOR`): stronger watermark energy
5. **Ensemble per-channel detection**: robust to channel-specific attacks

In [ ]:
import sys, importlib
from pathlib import Path

PROJECT_DIR = Path("/content/Invisible_Watermark_GNN")
TREE_RING_DIR = PROJECT_DIR / "tree-ring-watermark"
EXPERIMENT_DIR = PROJECT_DIR / "experiment"
for p in [str(TREE_RING_DIR), str(EXPERIMENT_DIR), str(PROJECT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import experiment.configs as configs
import experiment.run_optimized as run_optimized_mod
importlib.reload(configs)
importlib.reload(run_optimized_mod)

config_opt = (
    configs.get_small_scale_optimized()
    if SCALE == "small"
    else configs.get_large_scale_optimized()
)

print(f"Running OPTIMIZED ({SCALE} scale, {config_opt.end - config_opt.start} images, "
      f"{len(config_opt.attacks)} attacks, scale_factor={SCALE_FACTOR}) ...")
# results are also saved to experiment/results/ as JSON
optimized_results = run_optimized_mod.run_optimized(
    config_opt, skip_clip=SKIP_CLIP, scale_factor=SCALE_FACTOR
)
print("\n✅ Optimized experiment complete.")


---
## 📊 Cell 8 — Compare Results
Prints a side-by-side table of AUC, Accuracy, TPR@1%FPR, and per-attack metrics,  
then saves a CSV and bar-chart PNGs to `experiment/results/`.

In [ ]:
import sys, importlib
from pathlib import Path

PROJECT_DIR = Path("/content/Invisible_Watermark_GNN")
EXPERIMENT_DIR = PROJECT_DIR / "experiment"
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import experiment.compare_results as compare_mod
importlib.reload(compare_mod)

compare_mod.compare(SCALE)


---
## 📈 Cell 9 — Display Comparison Plots

In [ ]:
from pathlib import Path
from IPython.display import Image as IPyImage, display

plots_dir = EXPERIMENT_DIR / "results" / f"{SCALE}_plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"\n{p.name}")
        display(IPyImage(filename=str(p), width=800))
else:
    print(f"No plots found at {plots_dir}. Check that Cell 8 ran successfully.")

---
## 🖼️ Cell 10 — Display Sample Images
Shows watermarked vs. non-watermarked image pairs (first 3 images, no-attack condition).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from pathlib import Path

for approach in ["baseline", "optimized"]:
    out_dir = EXPERIMENT_DIR / "outputs" / f"{SCALE}_{approach}" / "no_attack"
    if not out_dir.exists():
        print(f"No saved images for {approach} (outputs dir not found).")
        continue

    wm_imgs   = sorted(out_dir.glob("*_wm.png"))[:3]
    no_wm_imgs = sorted(out_dir.glob("*_no_wm.png"))[:3]
    if not wm_imgs:
        print(f"No PNG images found in {out_dir}")
        continue

    n = min(len(wm_imgs), 3)
    fig, axes = plt.subplots(2, n, figsize=(5 * n, 11))
    fig.suptitle(
        f"{approach.upper()} — Watermarked (top) vs No-Watermark (bottom)",
        fontsize=14, fontweight="bold"
    )

    for j in range(n):
        axes[0, j].imshow(PILImage.open(wm_imgs[j]))
        axes[0, j].set_title(f"Watermarked {j}", fontsize=10)
        axes[0, j].axis("off")
        if j < len(no_wm_imgs):
            axes[1, j].imshow(PILImage.open(no_wm_imgs[j]))
            axes[1, j].set_title(f"No-Watermark {j}", fontsize=10)
        axes[1, j].axis("off")

    plt.tight_layout()
    plt.show()

---
## 💾 Cell 11 — Save Results to Google Drive *(Optional)*
Colab sessions are ephemeral. Run this cell to persist results, checkpoints, and output images  
to your Google Drive so they survive a runtime reset.

In [ ]:
#@title Save to Google Drive { display-mode: "form" }
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    import shutil
    from pathlib import Path

    drive_dst = Path("/content/drive/MyDrive/tree_ring_results")
    drive_dst.mkdir(parents=True, exist_ok=True)

    for folder in ["results", "checkpoints", "outputs"]:
        src = EXPERIMENT_DIR / folder
        dst = drive_dst / folder
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"  Copied {folder}/ → {dst}")
        else:
            print(f"  Skipping {folder}/ (not found)")

    print(f"\n✅ Results saved to {drive_dst}")
else:
    print("SAVE_TO_DRIVE is False — skipping Drive upload.")
    print("Set SAVE_TO_DRIVE = True and re-run this cell to save your results.")